# 7.10 答案被截光了還在訓練嗎？


一筆資料有很多token，可能全部都是問題。若模型窗口只留開頭兩格，就把後面的assistant與答案全截掉；輸入有數值，程式也能用目前模型的數字計算猜測，這一步叫前向，卻完全沒有要計分的答案目標。平均loss要把代價總和除以有效目標數；分母為零，就沒有可比較的平均值，不能記成0當成功。若工具顯示NaN，也就是「不是可用數字」，要先查這類資料問題。學習率控制每次調整模型數字的步幅；把步幅縮小不會增加被截掉的答案，無法修好這種缺少目標的情況。

[7.3](https://birdhackor.github.io/tiny-perceptron-vlm/7.3.html) 只讓assistant目標進入loss，[6.8](https://birdhackor.github.io/tiny-perceptron-vlm/6.8.html) 又規定窗口範圍。因此切短後必須重數有效label，逐筆確認仍至少有一個目標。即使保留部分答案，若EOS被截掉，也會改變結束訓練，見 [7.9](https://birdhackor.github.io/tiny-perceptron-vlm/7.9.html)。這兩種情況要分別記錄。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
from tiny_perceptron.data import render_chat, pad_batch

example = render_chat(
    [
        {"role": "user", "content": "很長的問題"},
        {"role": "assistant", "content": "是"},
    ]
)
try:
    pad_batch([example], max_length=2)
except ValueError as error:
    print("預期資料錯誤", error)
else:
    raise AssertionError("應該偵測空監督")
x, y = example
first = (y != -100).nonzero()[0].item()
print("首有效索引", first, "最低保留長度", first + 1, "完整長度", len(x))

輸入一筆中文問答，問題五字十五byte，答案一字三byte。`pad_batch` 把資料組batch，max_length=2只留前兩格，目標都-100，所以它明確報告第0筆截斷後沒有有效答案。`try` 表示試著執行，若出現指定ValueError就跑except；沒有預期錯誤則走else，並用AssertionError提醒檢查未生效。這段捕獲有意安排的錯誤，程序可繼續看後面的定位。

最後找首有效索引18，最低長度19才保留一個答案目標，完整長度22才含答的三byte與EOS。長度按X/Y格數計，不按中文字數；按 [7.4](https://birdhackor.github.io/tiny-perceptron-vlm/7.4.html) 首答案由assistant位置預測，所以最低合法長度不需要把全部答案都當輸入。合法有一格，只表示有分母，不保證完成了完整回答訓練。

處理長樣本可擴大窗口、重新設計切段、保留完整答案，或明確過濾無法使用的樣本。應保存原樣本編號與過濾原因，才能檢查是不是某類長題被系統性丟掉。只過濾錯誤不會自動提高能力，它只是讓訓練輸入符合目標契約。

這份工具還會逐筆拒絕無有效答案，避免一批裡有些筆全空卻被其他筆掩蓋。總體count大於0不是每筆都健康；應分項看來源、截斷比例與答案EOS保留率，來源統計可回看 [5.16](https://birdhackor.github.io/tiny-perceptron-vlm/5.16.html)。

自然對話尤其不能只問「放得進去嗎」。[T.4的UltraChat小型實跑](https://birdhackor.github.io/tiny-perceptron-vlm/training.html#T.4)另取100筆資料的首輪問題與回答，各保留最多120個byte、末尾保持完整字，再按原題家族切成80／10／10。這讓一筆能放進256格的小模型，也讓答案仍有可計分的位置；但有些問題與回答在句中就結束，意思可能已不完整。我們為片段加上的EOS，表示這次片段結束，不是原對話已回答完畢。

這個獨立模型只更新80次，十題驗證與十題最後檢查的答案ID匹配都是0，實際生成也有大量零碎字詞。開放式問答本來就不能只靠與示範一字不差判好壞；這份結果更沒有支持「已能完成原問題」。它適合確認下載、轉換、遮罩與更新走得通；若要研究完整對話能力，應先保留題目語意與完整答案，再安排相應模型和預算。

練習把max_length改為19，先預測不再報空監督、但只保留一個答案byte目標；把首次try塊改成直接接 `bx,by,valid=pad_batch(...)` 再印 `(by!=-100).sum()` 核對1。改成22應得到4。這樣用有效數與結束位置定位損失的內容，而不是只看長題有多少輸入token。
